# HelloDatabase Project
- Talk to your SQLDB using natural language.

In [36]:
import importlib.util

libs = ["langchain", "langchain_core", "langchain_experimental",
        "langchain_google_genai", "sqlalchemy", "pymysql",
        "mysql.connector", "dotenv"]

for lib in libs:
    print(f"{lib}: {'OK' if importlib.util.find_spec(lib) else 'MISSING'}")

langchain: OK
langchain_core: OK
langchain_experimental: OK
langchain_google_genai: OK
sqlalchemy: OK
pymysql: OK
mysql.connector: OK
dotenv: OK


In [37]:
#Checking if API Works and hence LLM Created

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0.2
)

# response = llm.invoke("Write a short poem on my love for dosa")
# print(response.content)


In [38]:
#Connection to SQL Workbench Databade

from langchain_community.utilities import SQLDatabase

db_user = "root"
db_password = "root"
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",
    sample_rows_in_table_info=4
)

print(db.get_usable_table_names())
print(db.table_info)

['discounts', 't_shirts']

CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
4 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
4	4	5.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
4 rows from t_shirts table:
t_sh

In [39]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)
qns1 = db_chain.invoke({"query": "How many t-shirts do we have left for Nike in extra small size and white color?"})
print(qns1)



> Entering new SQLDatabaseChain chain...
How many t-shirts do we have left for Nike in extra small size and white color?
SQLQuery:SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White';
SQLResult: [(Decimal('53'),)]
Answer:There are 53 Nike t-shirts left in extra small (XS) size and white color.
> Finished chain.
{'query': 'How many t-shirts do we have left for Nike in extra small size and white color?', 'result': 'There are 53 Nike t-shirts left in extra small (XS) size and white color.'}


In [40]:
qns1

{'query': 'How many t-shirts do we have left for Nike in extra small size and white color?',
 'result': 'There are 53 Nike t-shirts left in extra small (XS) size and white color.'}

In [41]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)
qns2 = db_chain.invoke({"query": "What is the total number of states in nepal?"})
print(qns2)



> Entering new SQLDatabaseChain chain...
What is the total number of states in nepal?
SQLQuery:SQLQuery: SELECT 'Data not available in the database' AS `message`;
SQLResult: [('Data not available in the database',)]
Answer:This information cannot be answered from the database, as the database only contains data related to t-shirts and discounts. However, in general knowledge, Nepal has 7 provinces (states).
> Finished chain.
{'query': 'What is the total number of states in nepal?', 'result': 'This information cannot be answered from the database, as the database only contains data related to t-shirts and discounts. However, in general knowledge, Nepal has 7 provinces (states).'}


In [42]:
qns2

{'query': 'What is the total number of states in nepal?',
 'result': 'This information cannot be answered from the database, as the database only contains data related to t-shirts and discounts. However, in general knowledge, Nepal has 7 provinces (states).'}

In [43]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)
qns3 = db_chain.invoke({"query": "What is the total price of the inventory for all Large size t-shirts?"})
print(qns3)



> Entering new SQLDatabaseChain chain...
What is the total price of the inventory for all Large size t-shirts?
SQLQuery:SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `size` = 'L';
SQLResult: [(Decimal('17236'),)]
Answer:The total price of the inventory for all Large size t-shirts is 17,236.
> Finished chain.
{'query': 'What is the total price of the inventory for all Large size t-shirts?', 'result': 'The total price of the inventory for all Large size t-shirts is 17,236.'}


In [44]:
qns3

{'query': 'What is the total price of the inventory for all Large size t-shirts?',
 'result': 'The total price of the inventory for all Large size t-shirts is 17,236.'}

In [51]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0.2
)

In [52]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)
qns4 = db_chain.invoke({"query": "For each t-shirt brand, calculate the total stock quantity, original inventory value, and inventory value after applying a 20% discount. Display only brands whose discounted inventory value exceeds ₹50,000."})
print(qns4)



> Entering new SQLDatabaseChain chain...
For each t-shirt brand, calculate the total stock quantity, original inventory value, and inventory value after applying a 20% discount. Display only brands whose discounted inventory value exceeds ₹50,000.
SQLQuery:```sql
SELECT `brand`, 
       SUM(`stock_quantity`) AS `total_stock_quantity`, 
       SUM(`price` * `stock_quantity`) AS `original_inventory_value`, 
       SUM(`price` * `stock_quantity` * 0.8) AS `discounted_inventory_value`
FROM `t_shirts`
GROUP BY `brand`
HAVING SUM(`price` * `stock_quantity` * 0.8) > 50000
LIMIT 5;
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT `brand`, \n       SUM(`stock_quantity`) AS `total_stock_quantity`, ' at line 1")
[SQL: ```sql
SELECT `brand`, 
       SUM(`stock_quantity`) AS `total_stock_quantity`, 
       SUM(`price` * `stock_quantity`) AS `original_inventory_value`, 
       SUM(`price` * `stock_quantity` * 0.8) AS `discounted_inventory_value`
FROM `t_shirts`
GROUP BY `brand`
HAVING SUM(`price` * `stock_quantity` * 0.8) > 50000
LIMIT 5;
```]
(Background on this error at: https://sqlalche.me/e/21/f405)

In [ ]:
qns4

In [188]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)
qns5 = db_chain.invoke({"query": "For each t-shirt brand, calculate the total stock quantity, original inventory value, and inventory value after applying a 20% discount. Display only brands whose discounted inventory value exceeds ₹20,000."})
print(qns5)
# Original ANS: Adidas	913	   27839 	22271.20



> Entering new SQLDatabaseChain chain...
For each t-shirt brand, calculate the total stock quantity, original inventory value, and inventory value after applying a 20% discount. Display only brands whose discounted inventory value exceeds ₹20,000.
SQLQuery:```sql
SELECT `brand`, 
       SUM(`stock_quantity`) AS `total_stock_quantity`, 
       SUM(`price` * `stock_quantity`) AS `original_inventory_value`, 
       SUM(`price` * `stock_quantity` * 0.8) AS `discounted_inventory_value`
FROM `t_shirts`
GROUP BY `brand`
HAVING `discounted_inventory_value` > 20000
LIMIT 5;
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT `brand`, \n       SUM(`stock_quantity`) AS `total_stock_quantity`, ' at line 1")
[SQL: ```sql
SELECT `brand`, 
       SUM(`stock_quantity`) AS `total_stock_quantity`, 
       SUM(`price` * `stock_quantity`) AS `original_inventory_value`, 
       SUM(`price` * `stock_quantity` * 0.8) AS `discounted_inventory_value`
FROM `t_shirts`
GROUP BY `brand`
HAVING `discounted_inventory_value` > 20000
LIMIT 5;
```]
(Background on this error at: https://sqlalche.me/e/21/f405)

In [ ]:
qns5

# Few Shot Learning Implement

In [102]:
# few_shots = [
#     {'Question' : "How many t-shirts do we have left for Nike in extra small size and white color?",
#      'SQLQuery' : "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike' AND `color` = 'White' AND `size` = 'XS';",
#      'SQLResult': "Result of the SQL query",
#      'Answer' : qns1},
    
#     {'Question': "What is the total number of states in nepal?",
#      'SQLQuery':"SELECT 'The provided database does not contain information about the states in Nepal.' AS `message`;",
#      'SQLResult': "Result of the SQL query",
#      'Answer': qns2},
    
#     {'Question': "What is the total price of the inventory for all Large sized t-shirts?" ,
#      'SQLQuery' : "SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `size` = 'L';",
#      'SQLResult': "Result of the SQL query",
#      'Answer': qns3} ,
    
#     #  {'Question' : "For each t-shirt brand, calculate the total stock quantity, original inventory value, and inventory value after applying a 20% discount. Display only brands whose discounted inventory value exceeds ₹50,000." ,
#     #   'SQLQuery': "SELECT `brand`, SUM(`stock_quantity`) AS `total_stock_quantity`, SUM(`price` * `stock_quantity`) AS `original_inventory_value`, SUM(`price` * `stock_quantity` * 0.8) AS `discounted_inventory_value` FROM `t_shirts` GROUP BY `brand` HAVING `discounted_inventory_value` > 50000 LIMIT 5;",
#     #   'SQLResult': "Result of the SQL query",
#     #   'Answer' : qns4},
   
#     # {'Question': "For each t-shirt brand, calculate the total stock quantity, original inventory value, and inventory value after applying a 20% discount. Display only brands whose discounted inventory value exceeds ₹20,000.",
#     #  'SQLQuery' : "SELECT brand, SUM(stock_quantity) AS total_stock_quantity, SUM(price * stock_quantity) AS original_inventory_value, SUM(price * stock_quantity) * 0.8 AS discounted_inventory_value FROM t_shirts GROUP BY brand HAVING discounted_inventory_value > 20000 LIMIT 5;,
#     #  'SQLResult': "Result of the SQL query",
#     #  'Answer' : qns5
#     #  }
# ]

# print("done")

In [235]:
few_shots = [
    {'Question' : "How many t-shirts do we have left for Nike in extra small size and white color?",
     'SQLQuery' : "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike' AND `color` = 'White' AND `size` = 'XS';",
     'SQLResult': "Result of the SQL query",
     'Answer' : qns1},
    
    {'Question': "What is the total number of states in nepal?",
     'SQLQuery':"SELECT 'The provided database does not contain information about the states in Nepal.' AS `message`;",
     'SQLResult': "Result of the SQL query",
     'Answer': qns2},
    
    {'Question': "What is the total price of the inventory for all Large sized t-shirts?" ,
     'SQLQuery' : "SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `size` = 'L';",
     'SQLResult': "Result of the SQL query",
     'Answer': qns3} 
]

In [236]:
for example in few_shots:
    if isinstance(example["Answer"], dict):
        example["Answer"] = example["Answer"]["result"]

## Creating Semantic Similarity Based example selector

- create embedding on the few shots
- Store the embeddings in Chroma DB
- Retrieve the top most Semantically close example from the vector store

In [ ]:
# %pip install -U langchain-huggingface sentence-transformers

In [237]:
# Testing HuggingFace Embeddings & Create Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# result = embeddings.embed_query("What is price of Nike S t-shirt?")
# print("Embedding dimensions:", len(result))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2512.07it/s]


In [223]:
# result[:8]

In [231]:
# print(few_shots)
# [example.values() for example in few_shots] 

In [241]:
# # Creating ChromaDB
to_vectorize = ["".join(example.values()) for example in few_shots]
to_vectorize[0] # Printing one example stored as a string

# # Creating ChromaDB
# to_vectorize = [" ".join(str(value) for value in example.values()) for example in few_shots]
# to_vectorize[0]  # Printing one example stored as a string

"How many t-shirts do we have left for Nike in extra small size and white color?SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike' AND `color` = 'White' AND `size` = 'XS';Result of the SQL queryThere are 53 Nike t-shirts left in extra small (XS) size and white color."

In [70]:
# %pip install langchain-chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\KIIT01\Desktop\LangChain\helloDatabase\lang-env\Scripts\python.exe -m pip install --upgrade pip


few_shots contains a nested dictionary in the Answer field, but ChromaDB metadata supports simple values, not nested dictionaries.
Fix: Convert nested metadata values to strings

Why this works:
- to_vectorize contains the text converted into embeddings.
- metadatas preserves your examples while converting nested dictionaries into JSON strings.
- ChromaDB can now store the metadata without raising the ValueError.

In [243]:
#Creating ChromaDB vector Database
from langchain_chroma import Chroma

vectorstore = Chroma.from_texts(
    texts=to_vectorize,
    embedding=embeddings,
    metadatas=few_shots
)

In [296]:
# Create the example selector
from langchain_classic.prompts import SemanticSimilarityExampleSelector

example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=2,
    input_keys=["input"]
)

example_selector.select_examples(
    {"input": "How many Adidas T shirts I have left in my store?"}
)


[{'Answer': 'There are 53 Nike t-shirts left in extra small (XS) size and white color.',
  'SQLQuery': "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike' AND `color` = 'White' AND `size` = 'XS';",
  'Question': 'How many t-shirts do we have left for Nike in extra small size and white color?',
  'SQLResult': 'Result of the SQL query'},
 {'Question': 'How many t-shirts do we have left for Nike in extra small size and white color?',
  'SQLResult': 'Result of the SQL query',
  'SQLQuery': "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike' AND `color` = 'White' AND `size` = 'XS';",
  'Answer': 'There are 53 Nike t-shirts left in extra small (XS) size and white color.'}]

# Setting Prompt

In [283]:
from langchain_classic.chains.sql_database.prompt import PROMPT_SUFFIX, _mysql_prompt
print(_mysql_prompt) #PROMPT_PREFIX

You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of

In [284]:
print(PROMPT_SUFFIX)

Only use the following tables:
{table_info}

Question: {input}


### Setting up PromptTemplete using input variables

In [297]:
from langchain_core.prompts import PromptTemplate

example_prompt = PromptTemplate(
    input_variables=["Question", "SQLQuery", "SQLResult", "Answer"],
    template="\nQuestion: {Question}\nSQLQuery: {SQLQuery}\nSQLResult: {SQLResult}\nAnswer: {Answer}"
)

In [298]:
#  Build the few-shot prompt
from langchain_core.prompts import FewShotPromptTemplate

few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector, #Hey LLM, if u're comfused, look in vectorDB
    example_prompt=example_prompt,
    prefix=mysql_prompt,
    suffix=PROMPT_SUFFIX,
    input_variables=["input", "table_info", "top_k"],
) 

print(few_shot_prompt)

input_variables=['input', 'table_info', 'top_k'] input_types={} partial_variables={} example_selector=SemanticSimilarityExampleSelector(vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002738C234AD0>, k=2, example_keys=None, input_keys=['input'], vectorstore_kwargs=None) example_prompt=PromptTemplate(input_variables=['Answer', 'Question', 'SQLQuery', 'SQLResult'], input_types={}, partial_variables={}, template='\nQuestion: {Question}\nSQLQuery: {SQLQuery}\nSQLResult: {SQLResult}\nAnswer: {Answer}') suffix='Only use the following tables:\n{table_info}\n\nQuestion: {input}' prefix='You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.\nUnless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informat

In [306]:
# # Recreate and run the SQL chain

# new_chain = SQLDatabaseChain.from_llm(
#     llm=llm, 
#     db=db,
#     verbose=True,
#     prompt=few_shot_prompt
# )

new_chain = SQLDatabaseChain.from_llm(
    llm=llm, 
    db=db,
    verbose=True,
    prompt=few_shot_prompt,
    return_intermediate_steps=True
)

In [307]:
# Test
print(few_shot_prompt.format(
    input="How many white color Levi's shirts I have?",
    table_info=db.get_table_info(),
    top_k=5
))



> Entering new SQLDatabaseChain chain...
How many white color Levi's shirts I have?
SQLQuery:GoogleRateLimitError Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 9h49m34.960439297s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/gene

In [308]:
new_chain.invoke({"query": "How many white color Levi's shirts I have?"})



> Entering new SQLDatabaseChain chain...
How many white color Levi's shirts I have?
SQLQuery:

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 9h27m11.632682313s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTie

In [ ]:
try:
    new_chain.invoke({"query": "How many white color Levi's shirts I have?"})
except Exception as e:
    print(getattr(e, "intermediate_steps", e))